# UMAP as a tensor problem: kNN graph, fuzzy sets and a force layout on MNIST

*ai · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/ai/umap_tensor)

UMAP, the dimension-reduction method that draws 784-pixel MNIST digits as a 2D map, written entirely as tensor operations: an exact kNN graph from one distance matrix and topk, a bisection that finds every point's bandwidth $\sigma_i$ at once, the fuzzy union $P + P^\top - P\circ P^\top$, a spectral start from the graph Laplacian, and attractive and repulsive forces applied to the whole edge list per epoch (by hand, and again by autograd). On 2500 digits and a Swiss roll it matches the umap-learn library on trustworthiness, neighbour recall and digit accuracy, far ahead of PCA.

**Vector form**

$$w = P + P^\top - P\circ P^\top,\qquad q_{ij} = \frac{1}{1 + a\,\lVert \mathbf y_i - \mathbf y_j\rVert^{2b}},\qquad C = \sum_{i\neq j} -\big[w_{ij}\log q_{ij} + (1-w_{ij})\log(1-q_{ij})\big]$$

**Index (tensor) form**

$$p_{j|i} = e^{-(d_{ij}-\rho_i)/\sigma_i},\ \ \sum_{j} p_{j|i} = \log_2 k \qquad w_{ij} = p_{j|i} + p_{i|j} - p_{j|i}\,p_{i|j} \qquad y_{ic} \mathrel{+}= \alpha \sum_{j} \frac{-2ab\, d_{ij}^{2(b-1)}}{1 + a\, d_{ij}^{2b}}\, (y_{ic}-y_{jc}) + \alpha \sum_{s} \frac{2b\,(y_{ic}-y_{sc})}{(\epsilon + d_{is}^2)(1 + a\, d_{is}^{2b})}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $d_{ij}$ | `knn(x, k)  # sq_dist + topk` | (N, k) | distances to the $k$ nearest points (the point itself first), from $\lVert x_i\rVert^2 + \lVert x_j\rVert^2 - 2\,x_{id}x_{jd}$ (repeated $d$ ⇒ summed) and topk; exact, row blocks keep memory at $O(BN)$ |
| $\rho_i,\ \sigma_i$ | `smooth_knn(dist, k)` | (N,), (N,) | $\rho_i$ = distance to the nearest other point; $\sigma_i$ by 64 halvings of one bracket tensor for all points at once, so that $\sum_j e^{-(d_{ij}-\rho_i)/\sigma_i} = \log_2 k$ |
| $p_{j\vert i}$ | `zeros(N, N).scatter_(1, idx, p)` | (N, N) | directed membership: 1 for the nearest neighbour, decaying with the distance beyond it, 0 outside the kNN list |
| $w_{ij}$ | `P + P.T - P * P.T` | (N, N) | fuzzy union (probabilistic OR) of the two directions: symmetric, in $[0, 1]$; its non-zeros are the edge list $(i_e, j_e)$, $E$ ≈ 50 000 for 2500 digits |
| $a,\ b$ | `fit_ab(min_dist)` | () | Gauss–Newton fit of $1/(1 + a x^{2b})$ to 1 below min_dist and $e^{-(x - \mathrm{min\_dist})}$ beyond: $a$ = 1.577, $b$ = 0.895 for min_dist = 0.1 |
| $L\,v = \lambda v$ | `linalg.eigh(I - D^-½ W D^-½)` | (N, N) | spectral start: the two lowest non-trivial eigenvectors of the normalised graph Laplacian, scaled to $[0, 10]$ |
| $\sum_{j},\ \sum_{s}$ | `out.index_add_(0, i, g)` | (N, 2) | forces of all sampled edges $(i, j)$ and of 5 random negatives $s$ per edge, summed per point in one scatter-add; each term clipped to $\pm 4$; learning rate $\alpha = 1 - \mathrm{epoch}/n$ |
| $-\partial C/\partial y_{ic}$ | `autograd.grad(sampled_loss(y, ...), y)` | (N, 2) | the same step from autograd on the sampled loss $\sum_e -\log q_{ij} + \sum_s -\log(1-q_{is})$: equal to the hand-written forces (tested) |
| $T(k)$ | `trustworthiness(x, y, 10)` | () | $1 - \frac{2}{Nk(2N-3k-1)}\sum_i\sum_{j\in U_k(i)}(r_{ij}-k)$, ranks $r$ from an argsort of an argsort; continuity swaps $x$ and $y$ |

The question. "UMAP as just a tensor problem" is Jacky Ka Long Ko's request: take UMAP (Uniform Manifold Approximation and Projection; McInnes, Healy and Melville 2018), usually run through the umap-learn library with its numba loops, and write every step as batched tensor operations, then check that the result is as good. Method, as tensors (N points in D dimensions). 1. kNN graph: the (N, N) squared distances $\lVert x_i\rVert^2 + \lVert x_j\rVert^2 - 2 x_i\cdot x_j$ (an einsum), computed in row blocks, then topk gives (N, k) distances and indices; exact, as umap-learn is below 4096 points (above that it switches to approximate nearest-neighbour descent). 2. Fuzzy simplicial set: $\rho_i$ is the distance to the nearest other point; $\sigma_i$ solves $\sum_j e^{-(d_{ij}-\rho_i)/\sigma_i} = \log_2 k$ by bisection, all N brackets halved together (64 times, a doubling phase first while the upper end is unknown); the memberships are scattered into a dense (N, N) matrix $P$ and symmetrised by the fuzzy union $W = P + P^\top - P\circ P^\top$ (dense is fine at a few thousand points; a sparse COO tensor is the same formula at scale). 3. The low-dimensional similarity $q = 1/(1 + a d^{2b})$, with $a$ and $b$ fitted to min_dist by a 2-parameter Gauss–Newton least-squares fit; start: the two lowest non-trivial eigenvectors of the normalised Laplacian $I - D^{-1/2} W D^{-1/2}$ from a dense torch.linalg.eigh (about 4 s for 2500 points on one core; umap-learn uses a sparse solver; if eigh fails the code falls back to a random start). 4. Layout: 500 epochs; an edge of weight $w_{ij}$ is active once every $\max w / w_{ij}$ epochs (umap-learn's schedule, kept as an (E,) tensor of next-sample epochs); every active edge attracts its two ends and draws 5 random negatives that repel its head; all forces of the epoch are computed at once and summed per point with index_add_ (a synchronous update, where umap-learn moves points edge by edge). The same loop with autograd: the gradient of the sampled cross-entropy, clipped after summing. 5. Metrics, all tensor ops with $k$ = 10: trustworthiness and continuity (Venna and Kaski 2001) from rank matrices (an argsort of an argsort), kNN recall, and leave-one-out 10-NN digit accuracy in the 2D layout; PCA by SVD as the linear baseline. Data. MNIST (LeCun, Cortes and Burges), licence CC BY-SA 3.0: data/mnist_subset.npz holds the first 250 training images of every digit (2500 images, uint8; the share-alike licence applies to this subset), see data/SOURCES.md. The Swiss roll (1500 points, as in scikit-learn's make_swiss_roll, with a little noise) is generated in code, so its true flat coordinates (arc length, height) are known. The umap-learn reference (umap-learn 0.5.12, scikit-learn 1.9.1, random_state 42) was computed once offline in a throwaway environment by data/make_reference.py; only its 2D coordinates and metrics are bundled. Results (bundled run, CPU, one thread). MNIST, 2500 digits: trustworthiness / continuity / 10-NN recall / 10-NN digit accuracy = 0.957 / 0.963 / 0.39 / 0.887 with the hand-written forces, 0.960 / 0.963 / 0.40 / 0.898 with autograd, 0.963 / 0.963 / 0.40 / 0.896 for umap-learn, and 0.740 / 0.913 / 0.07 / 0.414 for PCA. Swiss roll: trustworthiness 0.999 (umap-learn 0.999, PCA 0.943), and against the true flat coordinates 0.999 with 76 % of the true 10 nearest neighbours kept (umap-learn 77 %, PCA 28 %). Our trustworthiness also equals scikit-learn's to $2\times 10^{-4}$ on the bundled umap-learn layouts (tested), and $a$, $b$ equal umap-learn's. Wall time on MNIST: 11 s with the forces (0.5 s graph, 4 s eigh, 7 s layout), 15 s with autograd, 14 s for umap-learn (second call, one thread; its first call, with numba compilation, took 28 s). After the best rotation, reflection and scale (Procrustes), our MNIST layout differs from umap-learn's by a disparity of 0.05 (0 = same shape, 1 = unrelated); the same digit clusters sit in the same places. Verdict and limits. UMAP is a tensor problem: graph construction, bandwidth search, fuzzy union, spectral start and the force layout are all batched tensor operations, and the result is as good as the reference by every metric here. What differs: layouts are compared by metrics, never by coordinates, because UMAP layouts are only defined up to rotation, reflection and the randomness of negative sampling (another seed moves the clusters around); beyond 4096 points umap-learn's kNN becomes approximate while ours stays exact (and costs $O(N^2)$); our dense $N\times N$ matrices and eigh limit this version to about $10^4$ points (blocks, sparse tensors and an iterative eigensolver would lift that); the synchronous update makes early epochs look like a hot gas (the learning rate is near 1 and every point gets all its kicks at once) before the layout cools. The Swiss roll keeps every local neighbourhood but is not unrolled as one clean sheet: it tears into strips, as with umap-learn, a known trait of UMAP and t-SNE. The digit accuracy is a measure of how well the map separates classes it never saw, not a classifier. Results vary by a few thousandths across BLAS builds and thread counts. Related work. t-SNE (van der Maaten and Hinton 2008) set the pattern of matching neighbour probabilities with a heavy-tailed kernel in 2D; UMAP replaces its normalised probabilities with fuzzy memberships and its full repulsion with negative sampling (as in word2vec), which is why it scales. Laplacian eigenmaps (Belkin and Niyogi 2003) give the spectral start. Try it. Fewer neighbours (5) gives more, smaller clusters; min_dist near 0.9 packs points loosely; fewer epochs shows an unfinished layout; another seed rotates and reshuffles the map while the metrics stay put.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/ai/umap_tensor/main.py
"""Main loop: UMAP on an MNIST subset and on a Swiss roll, written as tensor operations; the same layout with
autograd forces; PCA as the linear baseline; quality metrics against the bundled umap-learn reference; tables and the
animation of the layout epochs."""

import csv
import time
from pathlib import Path

import torch

from problems._common.runtime import cli, device, fast_mode

from . import equation as E
from . import plot, solve
from . import tensors as T

K_METRIC = 10  # neighbourhood size of trustworthiness, continuity, recall and the k-NN classifier


def metrics(x, y, labels=None, flat=None) -> dict:
    """All quality numbers of one 2D layout y of data x (labels: digits; flat: true flat coordinates)."""
    m = {"trust": E.trustworthiness(x, y, K_METRIC), "cont": E.continuity(x, y, K_METRIC),
         "recall": E.knn_recall(x, y, K_METRIC)}  # fmt: skip
    if labels is not None:
        m["acc"] = E.knn_accuracy(y, labels, K_METRIC)
    if flat is not None:
        m["trust_flat"] = E.trustworthiness(flat, y, K_METRIC)
        m["recall_flat"] = E.knn_recall(flat, y, K_METRIC)
    return m


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    fast = fast_mode()
    k, md, seed = int(params["n_neighbors"]), float(params["min_dist"]), int(params["seed"])
    epochs = 40 if fast else int(params["n_epochs"])
    x, lab = T.mnist(dev, n=400 if fast else int(params["n_mnist"]))
    xs, flat, t_roll = T.swiss_roll(dev, n=400 if fast else T.N_ROLL)
    print(f"MNIST: {len(x)} images (balanced), X {tuple(x.shape)}; Swiss roll: X {tuple(xs.shape)}, true flat "
          f"coordinates U {tuple(flat.shape)}; device {dev}; k = {k}, min_dist = {md}, {epochs} epochs, "
          f"{params['neg_rate']} negatives per edge sample")  # fmt: skip
    a, b = E.fit_ab(md)
    print(f"curve fit: q = 1/(1 + a d^(2b)) with a = {a:.4f}, b = {b:.4f}")

    runs, rows = {}, []
    snaps = 5 if fast else plot.FRAMES
    for name, data, mode in (("mnist", x, "forces"), ("mnist", x, "autograd"), ("swiss_roll", xs, "forces")):
        prep = runs.get((name, "forces"))  # the autograd run reuses the graph and the spectral start
        o = solve.umap(data, k, md, epochs, seed, mode, snaps=snaps if mode == "forces" else 0,
                       neg_rate=int(params["neg_rate"]), prep=prep)  # fmt: skip
        runs[name, mode] = o
        s = o["seconds"]
        n_fail = int((o["sigma"] <= 1e-3 * o["dist"][:, 1:].mean(1) * 1.0001).sum())
        print(f"{name:10s} {mode:8s}: kNN + fuzzy set {s['graph']:.2f} s ({o['edges']} directed edges, σ at its "
              f"floor for {n_fail} points), spectral start {s['spectral']:.2f} s, layout {s['layout']:.2f} s, "
              f"total {s['total']:.2f} s")  # fmt: skip

    ref = T.reference()
    full = not fast and len(x) == 2500  # the reference was computed on the full bundled subset
    for name, data, extra in (("mnist", x, {"labels": lab}), ("swiss_roll", xs, {"flat": flat})):
        cands = [("PCA (SVD)", solve.pca(data), 0.0)]
        for mode in ("forces", "autograd"):
            if (name, mode) in runs:
                o = runs[name, mode]
                cands.append((f"tensor UMAP ({mode})", o["Y"], o["seconds"]["total"]))
        if ref is not None and (full or name == "swiss_roll") and not fast:
            warm = [float(r["seconds"]) for r in ref["rows"] if r["dataset"] == name and r["run"] == "1"]
            cands.append(("umap-learn (offline)", torch.from_numpy(ref[name]).to(dev), warm[0]))
        base = next((c[1] for c in cands if c[0].startswith("umap-learn")), None)
        for method, y, secs in cands:
            m = metrics(data, y, **extra)
            m["procrustes"] = solve.procrustes(y, base)[1] if base is not None else float("nan")
            rows.append({"dataset": name, "method": method, "seconds": secs, **m})
            txt = ", ".join(f"{key} {v:.3f}" for key, v in m.items())
            print(f"{name:10s} {method:22s}: {txt}" + (f", {secs:.1f} s" if secs else ""))

    res = out_dir / "results"
    res.mkdir(parents=True, exist_ok=True)
    keys = ["trust", "cont", "recall", "acc", "trust_flat", "recall_flat", "procrustes"]
    with open(res / "metrics.csv", "w", newline="", encoding="utf-8") as f:
        f.write(f"# umap_tensor: quality of each 2D layout, neighbourhood size k = {K_METRIC}. trust = "
                "trustworthiness, cont = continuity, recall = shared k-NN fraction, acc = leave-one-out 10-NN digit "
                "accuracy, *_flat = against the true flat Swiss-roll coordinates, procrustes = disparity to the "
                "umap-learn layout after the best rotation/reflection/scale (0 = same shape); seconds = wall time "
                f"(umap-learn: warm run, offline, single thread); device {dev}\n")  # fmt: skip
        w = csv.writer(f)
        w.writerow(["dataset", "method", "seconds", *keys])
        for r in rows:
            w.writerow([r["dataset"], r["method"], f"{r['seconds']:.3f}",
                        *(f"{r[k]:.4f}" if k in r else "" for k in keys)])  # fmt: skip
    o = runs["mnist", "forces"]
    with open(res / "mnist_embedding.csv", "w", newline="", encoding="utf-8") as f:
        f.write(
            "# umap_tensor: final tensor-UMAP layout (hand-written forces) of the MNIST subset; y1, y2 in "
            "arbitrary units; rho, sigma = the fuzzy-set parameters of each point (pixel-space units)\n"
        )
        w = csv.writer(f)
        w.writerow(["index", "digit", "y1", "y2", "rho", "sigma"])
        for i, (yy, d, r_, s_) in enumerate(zip(o["Y"].tolist(), lab.tolist(), o["rho"].tolist(),
                                                o["sigma"].tolist())):  # fmt: skip
            w.writerow([i, d, f"{yy[0]:.4f}", f"{yy[1]:.4f}", f"{r_:.4f}", f"{s_:.4f}"])

    t0 = time.perf_counter()
    plot.result_pngs(runs, rows, x, lab, xs, t_roll, ref if full else None, res)
    out = plot.animate(runs, lab, t_roll, epochs, out_dir / "umap_tensor.gif")
    print(f"figures {time.perf_counter() - t0:.1f} s")
    return out


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/ai/umap_tensor/equation.py
"""UMAP as tensor operations, one function per relation of the index form.

Shapes: N points, D input dimensions, k neighbours (the point itself included, as in umap-learn), E graph edges,
2 output dimensions. Summed indices are named in each docstring.
"""

import math

import torch


def sq_dist(x: torch.Tensor, z: torch.Tensor) -> torch.Tensor:
    """d²_ij = x_id x_id + z_jd z_jd - 2 x_id z_jd (repeated d ⇒ summed), (N, M), never below 0."""
    xx = torch.einsum("id,id->i", x, x)  # (N,)
    zz = torch.einsum("jd,jd->j", z, z)  # (M,)
    return (xx[:, None] + zz[None, :] - 2 * torch.einsum("id,jd->ij", x, z)).clamp(min=0)


def knn(x: torch.Tensor, k: int, block: int | None = None) -> tuple[torch.Tensor, torch.Tensor]:
    """Exact k nearest neighbours by brute force: all distances, then topk. dist, idx (N, k), the point itself first.
    block > 0 computes the (block, N) rows at a time, so memory is O(block·N) instead of O(N²)."""
    n = len(x)
    rows = range(0, n, block or n)
    d, ix = [], []
    for r0 in rows:  # one pass when block is None
        d2 = sq_dist(x[r0 : r0 + (block or n)], x)  # (B, N)
        own = torch.arange(r0, min(r0 + (block or n), n), device=x.device)
        d2[torch.arange(len(own), device=x.device), own] = -1.0  # the point itself always ranks first
        v, i = torch.topk(d2, k, dim=1, largest=False)  # (B, k), sorted ascending
        d.append(v.clamp(min=0).sqrt())
        ix.append(i)
    return torch.cat(d), torch.cat(ix)


def smooth_knn(dist: torch.Tensor, k: int, n_iter: int = 64) -> tuple[torch.Tensor, torch.Tensor]:
    """ρ_i = distance to the nearest other point; σ_i solves Σ_j exp(-(d_ij - ρ_i)/σ_i) = log2 k (sum over the k-1
    neighbours j ≠ i) by bisection on ALL points at once: one (N,) tensor of brackets, 64 halvings.
    Like umap-learn, σ_i is kept above 10⁻³ × the mean neighbour distance of i (never active on these data)."""
    d = dist[:, 1:]  # (N, k-1) neighbours without the point itself
    rho = torch.where(d > 0, d, torch.full_like(d, math.inf)).min(1).values  # (N,) first non-zero distance
    rho = torch.where(torch.isinf(rho), torch.zeros_like(rho), rho)
    target = math.log2(k)
    lo = torch.zeros_like(rho)
    hi = torch.full_like(rho, math.inf)
    sigma = torch.ones_like(rho)
    gap = (d - rho[:, None]).clamp(min=0)  # (N, k-1)
    for _ in range(n_iter):
        s = torch.exp(-gap / sigma[:, None]).sum(1)  # (N,) the sum for the current guesses
        too_big = s > target  # σ too large → shrink
        hi = torch.where(too_big, sigma, hi)
        lo = torch.where(too_big, lo, sigma)
        sigma = torch.where(torch.isinf(hi), 2 * sigma, (lo + hi) / 2)
    return rho, torch.maximum(sigma, 1e-3 * d.mean(1))


def membership(dist: torch.Tensor, rho: torch.Tensor, sigma: torch.Tensor) -> torch.Tensor:
    """p_{j|i} = exp(-(d_ij - ρ_i)/σ_i), (N, k); 1 for the nearest neighbour, 0 for the point itself."""
    p = torch.exp(-(dist - rho[:, None]).clamp(min=0) / sigma[:, None])
    p[:, 0] = 0.0
    return p


def fuzzy_union(p: torch.Tensor) -> torch.Tensor:
    """w_ij = p_ij + p_ji - p_ij p_ji: the probabilistic OR of the two directed memberships, (N, N) symmetric."""
    return p + p.T - p * p.T


def q(d2: torch.Tensor, a: float, b: float) -> torch.Tensor:
    """Low-dimensional similarity q_ij = 1 / (1 + a ‖y_i - y_j‖^{2b}), from the squared distance."""
    return 1.0 / (1.0 + a * d2.clamp(min=0) ** b)


def fit_ab(min_dist: float = 0.1, spread: float = 1.0, n_iter: int = 100) -> tuple[float, float]:
    """Least-squares fit of 1/(1 + a x^{2b}) to the target f(x) = 1 for x < min_dist, exp(-(x - min_dist)/spread)
    beyond, on 300 points in [0, 3·spread], by Gauss–Newton with Levenberg damping."""
    x = torch.linspace(0, 3 * spread, 300, dtype=torch.float64)[1:]  # x = 0 contributes 0 residual for any a, b
    f = torch.where(x < min_dist, torch.ones_like(x), torch.exp(-(x - min_dist) / spread))
    logx = torch.log(x)

    def residual(th):  # th = (a, b) → r (299,) and its Jacobian J (299, 2)
        u = th[0] * torch.exp(2 * th[1] * logx)  # a x^{2b}
        r = 1.0 / (1.0 + u) - f
        J = torch.stack([-u / th[0], -2 * logx * u], dim=1) / (1.0 + u)[:, None] ** 2
        return r, J

    th, lam = torch.tensor([1.0, 1.0], dtype=torch.float64), 1e-3
    for _ in range(n_iter):
        r, J = residual(th)
        step = torch.linalg.solve(J.T @ J + lam * torch.eye(2, dtype=th.dtype), -J.T @ r)
        new = th + step
        if (residual(new)[0] ** 2).sum() < (r**2).sum():
            th, lam = new, lam / 3
        else:
            lam *= 4
    return float(th[0]), float(th[1])


def forces(y, i, j, i_neg, k_neg, a, b, clip=4.0, eps=1e-3, w_att=None, w_rep=None):
    """Hand-written UMAP forces for the whole batch of sampled edges at once, (N, 2):
    attraction on edge (i, j):    f = -2ab d^{2(b-1)} / (1 + a d^{2b}) (y_i - y_j), added to i and subtracted from j;
    repulsion of negative (i, k): f = 2b / ((ε + d²)(1 + a d^{2b})) (y_i - y_k), added to i only.
    Each term is clipped to [-4, 4] per coordinate (umap-learn's clip); index_add_ sums the terms of a point."""
    out = torch.zeros_like(y)
    diff = y[i] - y[j]  # (E, 2)
    d2 = torch.einsum("ec,ec->e", diff, diff)  # (E,)
    c = torch.where(d2 > 0, -2 * a * b * d2.clamp(min=1e-30) ** (b - 1) / (1 + a * d2**b), torch.zeros_like(d2))
    g = c[:, None] * diff if w_att is None else (w_att * c)[:, None] * diff
    g = g if clip is None else g.clamp(-clip, clip)
    out.index_add_(0, i, g).index_add_(0, j, -g)
    diff = y[i_neg] - y[k_neg]  # (S, 2)
    d2 = torch.einsum("sc,sc->s", diff, diff)
    c = 2 * b / ((eps + d2) * (1 + a * d2**b))
    c = torch.where(i_neg == k_neg, torch.zeros_like(c), c)  # a point never repels itself
    g = c[:, None] * diff if w_rep is None else (w_rep * c)[:, None] * diff
    g = g if clip is None else g.clamp(-clip, clip)
    return out.index_add_(0, i_neg, g)


def sampled_loss(y, i, j, i_neg, k_neg, a, b, eps=0.0):
    """The cross-entropy UMAP samples: Σ_e -log q_ij (edges) + Σ_s -log(1 - q_ik) (negatives), with y_k detached so
    a negative sample moves only its head, as in umap-learn. ε shifts d² in the repulsive term (ε = 0: exact)."""
    d2 = ((y[i] - y[j]) ** 2).sum(1)  # (E,)
    att = torch.log1p(a * d2.clamp(min=1e-30) ** b)  # -log q
    d2n = ((y[i_neg] - y[k_neg].detach()) ** 2).sum(1) + eps  # (S,)
    keep = i_neg != k_neg
    u = a * d2n[keep] ** b
    rep = torch.log1p(u) - torch.log(u)  # -log(1 - q) = log((1 + u)/u)
    return att.sum() + rep.sum()


def cross_entropy(w: torch.Tensor, y: torch.Tensor, a: float, b: float) -> torch.Tensor:
    """Full fuzzy-set cross-entropy (without the constant entropy of w):
    C = Σ_{i≠j} -[w_ij log q_ij + (1 - w_ij) log(1 - q_ij)]  (double sum over i and j),
    computed as Σ_{i≠j} -log(1 - q_ij) over all pairs plus Σ_edges w_ij [log(1 - q_ij) - log q_ij] over the edges,
    with u = a d^{2b}: -log q = log(1 + u), -log(1 - q) = log(1 + 1/u)."""
    u = a * sq_dist(y, y).clamp(min=1e-12) ** b  # (N, N)
    rep = torch.log1p(1 / u)
    rep.fill_diagonal_(0.0)
    i, j = torch.nonzero(w, as_tuple=True)  # (E,) edges
    return rep.sum() + (w[i, j] * (torch.log1p(u[i, j]) - rep[i, j])).sum()


def _ranks(x: torch.Tensor) -> torch.Tensor:
    """r_ij = rank of j among i's neighbours by distance (1 = nearest), (N, N); argsort of argsort."""
    d2 = sq_dist(x, x)
    d2.fill_diagonal_(math.inf)
    return torch.argsort(torch.argsort(d2, dim=1), dim=1) + 1


def _knn_idx(x: torch.Tensor, k: int) -> torch.Tensor:
    return knn(x, k + 1)[1][:, 1:]  # (N, k) without the point itself


def trustworthiness(x: torch.Tensor, y: torch.Tensor, k: int) -> float:
    """T(k) = 1 - 2/(N k (2N - 3k - 1)) Σ_i Σ_{j ∈ U_k(i)} (r_ij - k): U_k(i) are the k nearest neighbours of i in
    the embedding y that are not among its k nearest in x, r_ij their rank in x (Venna & Kaski 2001)."""
    n = len(x)
    r = _ranks(x.double()).gather(1, _knn_idx(y.double(), k))  # (N, k) input ranks of embedding neighbours
    return 1 - 2 / (n * k * (2 * n - 3 * k - 1)) * float((r - k).clamp(min=0).sum())


def continuity(x: torch.Tensor, y: torch.Tensor, k: int) -> float:
    """C(k): trustworthiness with the roles swapped: input neighbours that the embedding pushed away."""
    return trustworthiness(y, x, k)


def knn_recall(x: torch.Tensor, y: torch.Tensor, k: int) -> float:
    """Fraction of the k nearest neighbours in x that are also among the k nearest in y."""
    n = len(x)
    in_x = torch.zeros(n, n, dtype=torch.bool, device=x.device)
    in_x.scatter_(1, _knn_idx(x.double(), k), True)
    return float(in_x.gather(1, _knn_idx(y.double(), k)).float().mean())


def knn_accuracy(y: torch.Tensor, labels: torch.Tensor, k: int) -> float:
    """Leave-one-out k-NN classification in the embedding: each point takes the majority label of its k nearest
    other points (votes = Σ_j one_hot(label_j) over the neighbours)."""
    votes = torch.nn.functional.one_hot(labels[_knn_idx(y.double(), k)], int(labels.max()) + 1).sum(1)  # (N, C)
    return float((votes.argmax(1) == labels).float().mean())


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/ai/umap_tensor/tensors.py
"""Build the concrete tensors: the bundled MNIST subset and the synthetic Swiss roll (a 2D sheet rolled up in 3D, so
its true flat coordinates are known), plus the bundled umap-learn reference embedding.

data/mnist_subset.npz holds `images` (2500, 28, 28) uint8 and `labels` (2500,) uint8: the first 250 images of every
digit in the MNIST training file, in file order. build_subset() made it once from the raw MNIST files, read with
gzip and numpy only (the raw folder is untrusted data, nothing in it is executed).
"""

import csv
import gzip
from pathlib import Path

import numpy as np
import torch

DATA = Path(__file__).parent / "data"
SUBSET = DATA / "mnist_subset.npz"
REF_NPZ = DATA / "umap_learn_reference.npz"
REF_CSV = DATA / "umap_learn_reference.csv"
PER_CLASS, CLASSES = 250, 10
N_ROLL, ROLL_SEED, ROLL_NOISE = 1500, 0, 0.05
MD5 = {  # the published checksums of the raw files (verified before use)
    "train-images-idx3-ubyte.gz": "f68b3c2dcbeaaa9fbdd348bbdeb94873",
    "train-labels-idx1-ubyte.gz": "d53e105ee54ea40749a09fcbcd1e9432",
}


def _idx(path: Path) -> np.ndarray:
    """Read one IDX file (big-endian header: magic, dims; then uint8 data) with gzip + numpy only."""
    raw = gzip.decompress(path.read_bytes())
    ndim = raw[3]
    shape = tuple(int.from_bytes(raw[4 + 4 * i : 8 + 4 * i], "big") for i in range(ndim))
    return np.frombuffer(raw, dtype=np.uint8, offset=4 + 4 * ndim).reshape(shape)


def build_subset(raw_dir: Path, out: Path = SUBSET) -> Path:
    """One-off: the first PER_CLASS training images of every digit, in file order."""
    import hashlib

    for name, md5 in MD5.items():
        assert hashlib.md5((raw_dir / name).read_bytes()).hexdigest() == md5, f"checksum mismatch: {name}"
    x, y = _idx(raw_dir / "train-images-idx3-ubyte.gz"), _idx(raw_dir / "train-labels-idx1-ubyte.gz")
    keep = np.sort(np.concatenate([np.flatnonzero(y == c)[:PER_CLASS] for c in range(CLASSES)]))
    np.savez_compressed(out, images=x[keep], labels=y[keep])
    return out


def mnist(device: torch.device, n: int | None = None) -> tuple[torch.Tensor, torch.Tensor]:
    """X (N, 784) pixels in [0, 1] (each image flattened to one row), labels (N,) int64. n keeps an evenly spread,
    still balanced subset (rows are in file order, digits interleaved)."""
    with np.load(SUBSET) as d:
        x = torch.from_numpy(d["images"].reshape(len(d["images"]), -1).astype(np.float32) / 255.0)
        y = torch.from_numpy(d["labels"].astype(np.int64))
    if n is not None and n < len(y):
        keep = torch.cat([torch.nonzero(y == c).flatten()[: n // CLASSES] for c in range(CLASSES)]).sort().values
        x, y = x[keep], y[keep]
    return x.to(device), y.to(device)


def swiss_roll_numpy(n: int = N_ROLL, seed: int = ROLL_SEED, noise: float = ROLL_NOISE) -> tuple:
    """The Swiss roll as in scikit-learn's make_swiss_roll, with numpy's PCG64 generator (stable across numpy
    versions, so the offline umap-learn reference sees exactly these points): roll angle t ∈ [1.5π, 4.5π],
    height h ∈ [0, 21]; x = t cos t, y = h, z = t sin t, plus Gaussian noise. Returns X (n, 3), t (n,), h (n,)."""
    rng = np.random.default_rng(seed)
    t = 1.5 * np.pi * (1 + 2 * rng.random(n))
    h = 21.0 * rng.random(n)
    x = np.stack([t * np.cos(t), h, t * np.sin(t)], axis=1) + noise * rng.standard_normal((n, 3))
    return x.astype(np.float32), t.astype(np.float32), h.astype(np.float32)


def swiss_roll(device: torch.device, n: int = N_ROLL) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """X (n, 3) points in space; U (n, 2) their true flat coordinates (arc length along the spiral, height);
    t (n,) roll angle (for colouring). The arc length s(t) = ∫ √(1 + t²) dt makes U an isometric unrolling."""
    x, t, h = (torch.from_numpy(a) for a in swiss_roll_numpy(n))
    s = 0.5 * (t * torch.sqrt(1 + t**2) + torch.asinh(t))  # arc length of the spiral r = t
    return x.to(device), torch.stack([s, h], dim=1).to(device), t.to(device)


def reference() -> dict | None:
    """The offline umap-learn run (data/make_reference.py): embeddings (N, 2) and its metric rows, or None."""
    if not REF_NPZ.is_file():
        return None
    with np.load(REF_NPZ) as d:
        out = {k: d[k] for k in d.files}
    with open(REF_CSV, encoding="utf-8") as f:
        rows = list(csv.DictReader(line for line in f if not line.startswith("#")))
    out["rows"] = rows
    return out


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/ai/umap_tensor/solve.py
"""The UMAP pipeline on tensors: kNN graph → fuzzy simplicial set → spectral start → layout optimisation, with the
forces written by hand (batched over the whole edge list) or taken from autograd on the sampled loss; plus PCA and
Procrustes alignment for the comparisons."""

import time

import torch

from problems._common.progress import steps

from . import equation as E


def fuzzy_graph(x: torch.Tensor, k: int, block: int | None = 1024) -> dict:
    """The high-dimensional graph: kNN (N, k), ρ and σ (N,), directed memberships P and their union W (N, N)."""
    dist, idx = E.knn(x, k, block=block)
    rho, sigma = E.smooth_knn(dist, k)
    p = E.membership(dist, rho, sigma)  # (N, k)
    P = torch.zeros(len(x), len(x), dtype=x.dtype, device=x.device).scatter_(1, idx, p)  # p_{j|i} placed at (i, j)
    return {"dist": dist, "idx": idx, "rho": rho, "sigma": sigma, "P": P, "W": E.fuzzy_union(P)}


def spectral_init(w: torch.Tensor, dim: int = 2, seed: int = 0) -> torch.Tensor:
    """The dim smallest non-trivial eigenvectors of the normalised graph Laplacian L = I - D^{-1/2} W D^{-1/2}
    (the trivial one, D^{1/2} 1, is shifted out of the way by adding 2 v0 v0ᵀ), mapped back by D^{-1/2} and
    rescaled to [0, 10] per axis as umap-learn does. torch.linalg.eigh is dense, O(N³): seconds for a few thousand
    points (umap-learn uses a sparse eigensolver). A disconnected graph has several zero eigenvalues; their
    eigenvectors are then component indicators, which simply place the components apart. Fallback if eigh fails: a
    uniform random start in [0, 10]², as umap-learn's init="random"."""
    g = torch.Generator(device="cpu").manual_seed(seed)
    n = len(w)
    try:
        dm = w.sum(1).clamp(min=1e-12).rsqrt()  # (N,) D^{-1/2}
        lap = torch.eye(n, dtype=w.dtype, device=w.device) - dm[:, None] * w * dm[None, :]
        v0 = 1 / dm / (1 / dm).norm()  # (N,) the trivial eigenvector D^{1/2} 1 (eigenvalue 0)
        _, vec = torch.linalg.eigh(lap + 2 * v0[:, None] * v0[None, :])  # trivial one moved to the top, (N, N)
        y = dm[:, None] * vec[:, :dim]  # Laplacian eigenmaps: L_rw y = λ y with y = D^{-1/2} v
    except torch.linalg.LinAlgError:
        print("spectral initialisation failed; random start instead")
        y = torch.rand(n, dim, generator=g).to(w)
    y = y + 1e-4 * (y.abs().max()) * torch.randn(n, dim, generator=g).to(w)  # break exact ties
    return 10 * (y - y.min(0).values) / (y.max(0).values - y.min(0).values)


def edge_list(w: torch.Tensor, n_epochs: int) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Heads, tails (E,) of the non-zero memberships (both directions) and epochs per sample max(w)/w_ij: an edge of
    weight w_ij is sampled w_ij/max(w) times per epoch on average; edges too weak to be sampled once are dropped."""
    i, j = torch.nonzero(w, as_tuple=True)
    wij = w[i, j]
    keep = wij >= wij.max() / n_epochs
    return i[keep], j[keep], wij.max() / wij[keep]


def optimise(y0, i, j, eps, a, b, n_epochs, neg_rate=5, seed=0, mode="forces", w=None, snaps=0):
    """SGD on the layout. Each epoch every edge whose turn has come is active (a (E,) mask: the same schedule as
    umap-learn), each active edge draws neg_rate random negatives, and ALL points move at once (a Jacobi update of
    the batch, where umap-learn updates edge by edge). Learning rate α = 1 - epoch/n_epochs.
    mode "forces": the hand-written force formula; mode "autograd": the gradient of the same sampled loss from
    autograd, clipped per coordinate to ±4 after summing. Returns Y (N, 2), snapshots [(epoch, Y, C)], seconds."""
    g = torch.Generator(device="cpu").manual_seed(seed)
    n = len(y0)
    y = y0.clone()
    nxt = eps.clone()  # epoch of the next sample of each edge
    every = max(1, n_epochs // snaps) if snaps else 0
    frames = [(0, y.cpu().clone())] if snaps else []
    t0 = time.perf_counter()
    for ep in steps(n_epochs, f"layout ({mode})"):
        alpha = 1.0 - ep / n_epochs
        act = nxt <= ep  # (E,) edges sampled this epoch
        ia, ja = i[act], j[act]
        i_neg = ia.repeat_interleave(neg_rate)  # (S,) heads of the negative samples
        k_neg = torch.randint(0, n, (len(i_neg),), generator=g).to(y.device)  # (S,) random tails
        if mode == "forces":
            y += alpha * E.forces(y, ia, ja, i_neg, k_neg, a, b)
        else:
            yy = y.clone().requires_grad_(True)
            (grad,) = torch.autograd.grad(E.sampled_loss(yy, ia, ja, i_neg, k_neg, a, b, eps=1e-3), yy)
            y -= alpha * grad.clamp(-4, 4)
        nxt[act] += eps[act]
        if every and ((ep + 1) % every == 0 or ep + 1 == n_epochs):
            frames.append((ep + 1, y.cpu().clone()))
    secs = time.perf_counter() - t0
    frames = [(e, f, float(E.cross_entropy(w, f.to(w.device), a, b))) for e, f in frames]  # untimed: for the plot
    return y, frames, secs


def umap(x, k=15, min_dist=0.1, n_epochs=500, seed=0, mode="forces", snaps=0, neg_rate=5, prep=None) -> dict:
    """The whole pipeline on (N, D) data; returns the layout and every intermediate, with wall time per stage.
    prep: an earlier result on the same data whose graph and spectral start are reused (with their times)."""
    torch.manual_seed(seed)
    if prep is None:
        t = {}
        t0 = time.perf_counter()
        graph = fuzzy_graph(x, k)
        t["graph"] = time.perf_counter() - t0
        t0 = time.perf_counter()
        y0 = spectral_init(graph["W"], 2, seed)
        t["spectral"] = time.perf_counter() - t0
    else:
        graph = {key: prep[key] for key in ("dist", "idx", "rho", "sigma", "P", "W")}
        y0, t = prep["Y0"], {key: prep["seconds"][key] for key in ("graph", "spectral")}
    a, b = E.fit_ab(min_dist)
    i, j, eps = edge_list(graph["W"], n_epochs)
    y, frames, t["layout"] = optimise(y0, i, j, eps, a, b, n_epochs, neg_rate, seed, mode, graph["W"], snaps)
    t["total"] = t["graph"] + t["spectral"] + t["layout"]
    return {"Y": y, "Y0": y0, "frames": frames, "a": a, "b": b, "edges": len(i), "seconds": t, **graph}


def expected_step(y, w, a, b, lr):
    """One step along the EXPECTED force: every ordered pair attracts with weight w_ij and repels with weight
    2(1 - w_ij) (the repulsion of a pair moves only its head, the reversed pair moves the other end). Without
    clipping this is exactly -lr ∂C/∂y of the full cross-entropy."""
    n = len(y)
    i, j = torch.nonzero(~torch.eye(n, dtype=torch.bool, device=y.device), as_tuple=True)
    wij = w[i, j]
    return y + lr * E.forces(y, i, j, i, j, a, b, clip=None, eps=0.0, w_att=wij, w_rep=2 * (1 - wij))


def pca(x: torch.Tensor, dim: int = 2) -> torch.Tensor:
    """Projection on the first principal axes: X_c = X - mean = U S Vᵀ (SVD), coordinates X_c V[:, :dim]."""
    xc = x - x.mean(0)
    _, _, vh = torch.linalg.svd(xc, full_matrices=False)
    return xc @ vh[:dim].T


def procrustes(y: torch.Tensor, ref: torch.Tensor) -> tuple[torch.Tensor, float]:
    """Align y to ref by translation, uniform scale, rotation and reflection (orthogonal R from the SVD of
    y_cᵀ ref_c); both normalised to unit Frobenius norm. Returns y aligned in ref's frame and the disparity
    ‖y R - ref‖² ∈ [0, 1] (0 = same shape)."""
    yc, rc = y - y.mean(0), ref - ref.mean(0)
    ny, nr = yc.norm(), rc.norm()
    u, s, vh = torch.linalg.svd((yc / ny).T @ (rc / nr))
    r = u @ vh  # (2, 2) orthogonal
    disparity = float(1 - s.sum() ** 2)
    return (yc / ny) @ r * s.sum() * nr + ref.mean(0), disparity


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/ai/umap_tensor/plot.py
"""The animation: the MNIST layout and the Swiss roll unrolling, epoch by epoch, from the spectral start; and the
static result panels: metrics against PCA and umap-learn, ρ/σ histograms, the fuzzy kNN graph, the final layouts."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.collections import LineCollection
from matplotlib.colors import Normalize
from mpl_toolkits.mplot3d.art3d import Line3DCollection
from PIL import Image

from problems._common.plotting import export_plot_data, save_png
from problems._common.progress import report

from . import solve

FRAMES = 50  # layout snapshots per run (plus the spectral start)
RC = {"axes.titlesize": 12, "axes.labelsize": 11, "xtick.labelsize": 10, "ytick.labelsize": 10, "legend.fontsize": 10}
DIGITS = plt.get_cmap("tab10")  # digit d → DIGITS(d), the same in every figure
ROLL = Normalize(1.5 * np.pi, 4.5 * np.pi)  # roll angle t → viridis
METHOD_COLOUR = {"PCA (SVD)": "#9ca3af", "tensor UMAP (forces)": "#2563eb", "tensor UMAP (autograd)": "#16a34a",
                 "umap-learn (offline)": "#dc2626"}  # fmt: skip


def save_gif_shared_palette(fig, update, frames: list, path: Path, durations: list[int]) -> Path:
    """Every frame quantised with ONE palette built from a strip of frames, so colours do not drift between frames;
    durations (ms) per frame let the last frame stay on screen."""
    fig.set_dpi(80)
    shots = []
    for i, f in enumerate(frames):
        update(f)
        fig.canvas.draw()
        shots.append(Image.fromarray(np.asarray(fig.canvas.buffer_rgba())[..., :3]))
        report(i + 1, len(frames), "frames")
    export_plot_data(fig, path)
    plt.close(fig)
    sample = shots[:: max(1, len(shots) // 12)]
    strip = Image.new("RGB", (sample[0].width, sample[0].height * len(sample)))
    for k, im in enumerate(sample):
        strip.paste(im, (0, k * im.height))
    palette = strip.quantize(colors=255, method=Image.Quantize.MEDIANCUT)
    frames_p = [im.quantize(palette=palette, dither=Image.Dither.NONE) for im in shots]
    tmp = path.with_name(f".{path.stem}.tmp{path.suffix}")
    frames_p[0].save(tmp, save_all=True, append_images=frames_p[1:], duration=durations, loop=0)
    return tmp.replace(path)


def _limits(ys: list[np.ndarray]) -> tuple:
    """Axis limits covering every frame (5 % margin), fixed for the whole animation."""
    lo = np.min([y.min(0) for y in ys], axis=0)
    hi = np.max([y.max(0) for y in ys], axis=0)
    pad = 0.05 * (hi - lo)
    return (lo[0] - pad[0], hi[0] + pad[0]), (lo[1] - pad[1], hi[1] + pad[1])


def _digit_legend(ax, **kw):
    handles = [plt.Line2D([], [], ls="", marker="o", ms=7, color=DIGITS(d), label=str(d)) for d in range(10)]
    ax.legend(handles=handles, title="digit", **kw)


def animate(runs: dict, lab, t_roll, epochs: int, path: Path) -> Path:
    fm, fs = runs["mnist", "forces"]["frames"], runs["swiss_roll", "forces"]["frames"]
    ym = [f[1].numpy() for f in fm]
    ysr = [f[1].numpy() for f in fs]
    cm, cs = lab.cpu().numpy(), t_roll.cpu().numpy()
    with plt.rc_context(RC):
        fig = plt.figure(figsize=(13, 5.6))
        ax1 = fig.add_axes([0.06, 0.12, 0.36, 0.72])
        ax2 = fig.add_axes([0.55, 0.12, 0.36, 0.72])
        cax = fig.add_axes([0.925, 0.12, 0.015, 0.72])
        sc1 = ax1.scatter(ym[0][:, 0], ym[0][:, 1], c=[DIGITS(d) for d in cm], s=5, linewidths=0, label="digits")
        sc2 = ax2.scatter(ysr[0][:, 0], ysr[0][:, 1], c=cs, cmap="viridis", norm=ROLL, s=5, linewidths=0,
                          label="Swiss roll points")  # fmt: skip
        for ax, ys in ((ax1, ym), (ax2, ysr)):
            xl, yl = _limits(ys)
            ax.set_xlim(*xl)
            ax.set_ylim(*yl)
            ax.set_xlabel("$y_1$ (a.u.)")
            ax.set_ylabel("$y_2$ (a.u.)")
        _digit_legend(ax1, loc="upper left", bbox_to_anchor=(1.01, 1.0), fontsize=10, title_fontsize=10,
                      handletextpad=0.2, borderaxespad=0.2)  # fmt: skip
        cb = fig.colorbar(sc2, cax=cax)
        cb.set_label("roll angle $t$ (rad)")
        sup = fig.suptitle("", fontsize=13)

        def update(k):
            ep, _, c1 = fm[k]
            sc1.set_offsets(ym[k])
            sc2.set_offsets(ysr[k])
            ax1.set_title(f"MNIST, {len(cm)} digits\ncross-entropy $C$ = {c1:.3g} (a.u.)")
            ax2.set_title(f"Swiss roll unrolling, {len(cs)} points\ncross-entropy $C$ = {fs[k][2]:.3g} (a.u.)")
            sup.set_text(f"UMAP as tensors: epoch {ep} / {epochs}" + ("  (spectral start)" if ep == 0 else ""))

        n = len(fm)
        durations = [1200] + [100] * (n - 2) + [3000] if n > 1 else [3000]
        return save_gif_shared_palette(fig, update, list(range(n)), path, durations)


def _bars(ax, rows, dataset, keys, names):
    meth = [r for r in rows if r["dataset"] == dataset]
    width = 0.8 / len(meth)
    for m, r in enumerate(meth):
        xs = np.arange(len(keys)) + (m - (len(meth) - 1) / 2) * width
        vals = [r.get(k, np.nan) for k in keys]
        ax.bar(xs, vals, width, color=METHOD_COLOUR[r["method"]], label=r["method"])
        for xx, v in zip(xs, vals):
            ax.text(xx, v + 0.01, f"{v:.2f}", ha="center", va="bottom", fontsize=9, rotation=90)
    ax.set_xticks(np.arange(len(keys)), names)
    ax.set_ylim(0, 1.18)
    ax.set_ylabel("score (dimensionless)")


def result_pngs(runs, rows, x, lab, xs, t_roll, ref, res: Path) -> None:
    with plt.rc_context(RC):
        # 1. metrics vs PCA vs umap-learn, and wall time
        fig, axs = plt.subplots(1, 3, figsize=(16, 5.2), gridspec_kw={"width_ratios": [1.2, 1.2, 0.9]})
        _bars(axs[0], rows, "mnist", ["trust", "cont", "recall", "acc"],
              ["trustw.", "contin.", "recall@10", "10-NN acc."])  # fmt: skip
        axs[0].set_title("MNIST subset ($k$ = 10)")
        _bars(axs[1], rows, "swiss_roll", ["trust", "cont", "trust_flat", "recall_flat"],
              ["trustw.", "contin.", "trustw.\nvs flat", "recall@10\nvs flat"])  # fmt: skip
        axs[1].set_title("Swiss roll ($k$ = 10)")
        h, lb = axs[0].get_legend_handles_labels()
        fig.legend(h, lb, loc="upper center", ncol=4, fontsize=10, frameon=False)
        timed = [r for r in rows if r["seconds"] > 0]
        short = {"tensor UMAP (forces)": "tensor, forces", "tensor UMAP (autograd)": "tensor, autograd",
                 "umap-learn (offline)": "umap-learn"}  # fmt: skip
        labels = [f"{short[r['method']]}\n{'MNIST' if r['dataset'] == 'mnist' else 'Swiss roll'}" for r in timed]
        axs[2].barh(np.arange(len(timed)), [r["seconds"] for r in timed],
                    color=[METHOD_COLOUR[r["method"]] for r in timed])  # fmt: skip
        axs[2].set_yticks(np.arange(len(timed)), labels, fontsize=10)
        axs[2].invert_yaxis()
        axs[2].set_xlim(0, 1.2 * max(r["seconds"] for r in timed))
        axs[2].set_xlabel("wall time (s)")
        axs[2].set_title("time to embed (CPU)")
        fig.tight_layout(rect=(0, 0, 1, 0.92))
        save_png(fig, res / "metrics.png")

        # 2. ρ and σ histograms
        fig, axs = plt.subplots(2, 2, figsize=(11, 7.5))
        for row, name in enumerate(("mnist", "swiss_roll")):
            o = runs[name, "forces"]
            for col, (key, sym) in enumerate((("rho", r"$\rho_i$"), ("sigma", r"$\sigma_i$"))):
                v = o[key].cpu().numpy()
                axs[row, col].hist(v, bins=40, color="#2563eb" if col == 0 else "#ea580c")
                axs[row, col].set_xlabel(f"{sym}, " + ("pixel-space distance (a.u.)" if name == "mnist"
                                                       else "distance (a.u.)"))  # fmt: skip
                axs[row, col].set_ylabel("points (count)")
                axs[row, col].set_title(f"{'MNIST' if name == 'mnist' else 'Swiss roll'}: {sym}, median "
                                        f"{np.median(v):.3g}")  # fmt: skip
        fig.tight_layout()
        save_png(fig, res / "rho_sigma.png")

        # 3. the fuzzy kNN graph on small samples
        fig = plt.figure(figsize=(13, 5.6))
        ax = fig.add_subplot(1, 2, 1, projection="3d")
        g = torch.Generator().manual_seed(0)
        pick = torch.randperm(len(xs), generator=g)[:300].to(xs.device)
        sub = xs[pick]
        gr = solve.fuzzy_graph(sub, 10)
        i, j = torch.nonzero(torch.triu(gr["W"]), as_tuple=True)
        p = sub.cpu().numpy()
        w = gr["W"][i, j].cpu().numpy()
        wn = Normalize(0, 1)
        lc3 = Line3DCollection(np.stack([p[i.cpu().numpy()], p[j.cpu().numpy()]], axis=1), cmap="magma_r", norm=wn,
                               linewidths=0.8)  # fmt: skip
        lc3.set_array(w)
        ax.add_collection(lc3)
        sc = ax.scatter(p[:, 0], p[:, 1], p[:, 2], c=t_roll[pick].cpu().numpy(), cmap="viridis", norm=ROLL, s=10,
                        label="sampled points")  # fmt: skip
        fig.colorbar(sc, ax=ax, shrink=0.7, pad=0.12).set_label("roll angle $t$ (rad)")
        ax.set_xlabel("$x_1$ (a.u.)")
        ax.set_ylabel("$x_2$ (a.u.)")
        ax.set_zlabel("$x_3$ (a.u.)")
        ax.set_title("Swiss roll, 300 points: fuzzy 10-NN graph in 3D")
        ax2 = fig.add_subplot(1, 2, 2)
        o = runs["mnist", "forces"]
        y = o["Y"].cpu().numpy()
        ax2.scatter(y[:, 0], y[:, 1], c=[DIGITS(d) for d in lab.cpu().numpy()], s=3, alpha=0.35, linewidths=0,
                    label="all digits")  # fmt: skip
        src = torch.randperm(len(y), generator=g)[:40]
        W = o["W"].cpu()
        segs, ws = [], []
        for s_ in src.tolist():
            nb = torch.nonzero(W[s_]).flatten().tolist()
            segs += [(y[s_], y[t_]) for t_ in nb]
            ws += [float(W[s_, t_]) for t_ in nb]
        lc = LineCollection(segs, cmap="magma_r", norm=wn, linewidths=0.9)
        lc.set_array(np.array(ws))
        ax2.add_collection(lc)
        cb = fig.colorbar(lc, ax=ax2)
        cb.set_label("membership $w_{ij}$ (dimensionless)")
        ax2.set_xlabel("$y_1$ (a.u.)")
        ax2.set_ylabel("$y_2$ (a.u.)")
        ax2.set_title("MNIST layout: all graph edges of 40 points")
        fig.tight_layout()
        save_png(fig, res / "knn_graph.png")

        # 4. final layouts side by side
        y_ours = runs["mnist", "forces"]["Y"].cpu()
        panels = [("tensor UMAP (forces)", y_ours), ("tensor UMAP (autograd)", runs["mnist", "autograd"]["Y"].cpu())]
        if ref is not None:
            yr, disp = solve.procrustes(torch.from_numpy(ref["mnist"]), y_ours)
            panels.append((f"umap-learn, Procrustes-aligned\n(disparity {disp:.2f})", yr))
        panels.append(("PCA (first two axes)", solve.pca(x).cpu()))
        fig, axs = plt.subplots(1, len(panels), figsize=(4.4 * len(panels) + 1, 4.9))
        c = [DIGITS(d) for d in lab.cpu().numpy()]
        for ax, (title, yy) in zip(axs, panels):
            ax.scatter(yy[:, 0].numpy(), yy[:, 1].numpy(), c=c, s=3, linewidths=0, label=title.split("\n")[0])
            ax.set_title(title)
            ax.set_xlabel("$y_1$ (a.u.)")
            ax.set_ylabel("$y_2$ (a.u.)")
        _digit_legend(axs[-1], loc="upper left", bbox_to_anchor=(1.01, 1.0), fontsize=10, title_fontsize=10)
        fig.tight_layout()
        save_png(fig, res / "embeddings.png")


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'n_neighbors': 15,
    'min_dist': 0.1,
    'n_epochs': 500,
    'neg_rate': 5,
    'n_mnist': 2500,
    'seed': 0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.ai.umap_tensor")]:
    del sys.modules[name]
main = importlib.import_module("problems.ai.umap_tensor.main")
problem_dir = Path("problems/ai/umap_tensor")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
